In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
base_path = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath"

dataset_path = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/questions_w_answers.jsonl"

mcq_input_path = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath_Questions_163_189_without_Correct_Answer.csv"

mcq_answer_path = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath_Questions_163_189_with_Correct_Answer.csv"

In [13]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00


In [14]:
import torch
import json
import pandas as pd
import os

In [15]:
def load_moaath_open(file_path):
    questions = []
    with open(file_path, "r") as f:
        for i, line in enumerate(f):
            if 100 <= i <= 116:
                data = json.loads(line)
                questions.append({
                    "id": i + 1,
                    "question": data["Question"],
                    "must_have": data["Must_have"]
                })
    return questions

open_questions = load_moaath_open(dataset_path)
print("Open Questions:", len(open_questions))

Open Questions: 17


In [16]:
mcq_input = pd.read_csv(mcq_input_path)
mcq_answers = pd.read_csv(mcq_answer_path)

print("MCQ Questions:", len(mcq_input))

MCQ Questions: 27


In [17]:
def extract_choice(text):
    text = text.upper()

    for c in ["A","B","C","D","E","F"]:
        patterns = [
            f"{c})",
            f"{c}.",
            f"ANSWER: {c}",
            f"OPTION {c}",
            f" {c} "
        ]

        for p in patterns:
            if p in text:
                return c

    return None

In [18]:
def compute_must_have(answer, must_have_list):
    answer = answer.lower()
    found = 0

    for item in must_have_list:
        words = item.lower().split()
        match = sum(1 for w in words if w in answer)

        if match >= max(1, len(words)//2):
            found += 1

    if len(must_have_list) == 0:
        return 0

    return found / len(must_have_list)

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Mistral loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral loaded


In [20]:
open_results = []

for q in open_questions:

    prompt = f"""You are a medical expert.
Answer clearly:

{q['question']}
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    score = compute_must_have(response, q["must_have"])

    open_results.append({
        "id": q["id"],
        "question": q["question"],
        "answer": response,
        "must_have_score": score
    })

open_df = pd.DataFrame(open_results)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for ope

In [21]:
open_df.to_csv(
    f"{base_path}/mistral_open_moaath.csv",
    index=False
)

In [40]:
model_name = "mistral"   # غيّر الاسم حسب النموذج

mcq_results = []

for i,row in mcq_input.iterrows():

    prompt = f"""Answer with only one letter (A,B,C,D,E or F).

{row['Question_text']}

A) {row['(A)']}
B) {row['(B)']}
C) {row['(C)']}
D) {row['(D)']}
E) {row['(E)']}
F) {row['(F)']}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=1,
        do_sample=False
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    response = response.strip().upper()

    pred = response[-1] if len(response) > 0 else None
    correct = mcq_answers.loc[i,"Correct_answer"].strip().upper()

    mcq_results.append({
        "question": row["Question_text"],
        f"{model_name}_prediction": pred,
        "correct": correct,
        "score": pred == correct
    })

mcq_df = pd.DataFrame(mcq_results)

accuracy = mcq_df["score"].mean()
print("Accuracy:", accuracy)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

Accuracy: 0.3333333333333333


In [41]:
mcq_df.to_csv(
    f"{base_path}/mistral_mcq_moaath.csv",
    index=False
)

In [50]:
model_name = "mistral"

# MCQ
mcq_accuracy = mcq_df["score"].mean()
mcq_correct = mcq_df["score"].sum()
mcq_total = len(mcq_df)

# OPEN
open_mean = open_df["must_have_score"].mean()
open_median = open_df["must_have_score"].median()
open_std = open_df["must_have_score"].std()
open_min = open_df["must_have_score"].min()
open_max = open_df["must_have_score"].max()

open_good = (open_df["must_have_score"] >= 0.5).sum()
open_excellent = (open_df["must_have_score"] >= 0.8).sum()

results_df = pd.DataFrame([{
    "model": model_name,
    "mcq_accuracy": mcq_accuracy,
    "mcq_correct": mcq_correct,
    "mcq_total": mcq_total,
    "open_mean": open_mean,
    "open_median": open_median,
    "open_std": open_std,
    "open_min": open_min,
    "open_max": open_max,
    "open_good_0.5": open_good,
    "open_excellent_0.8": open_excellent
}])

csv_path = f"{base_path}/{model_name}_summary.csv"

results_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
print(results_df)

Saved: /content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath/mistral_summary.csv
     model  mcq_accuracy  mcq_correct  mcq_total  open_mean  open_median  \
0  mistral      0.333333            9         27   0.611928          0.6   

   open_std  open_min  open_max  open_good_0.5  open_excellent_0.8  
0  0.372073       0.0       1.0             11                   7  


In [54]:
import gc
import torch

try:
    del model
except:
    pass

try:
    del tokenizer
except:
    pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [55]:
torch.cuda.memory_summary()

'|===========================================================================|\n|                  PyTorch CUDA memory summary, device ID 0                 |\n|---------------------------------------------------------------------------|\n|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |\n|===========================================================================|\n|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |\n|---------------------------------------------------------------------------|\n| Allocated memory      |   8333 KiB |  12642 MiB |   5507 GiB |   5507 GiB |\n|       from large pool |   8320 KiB |  12627 MiB |   4906 GiB |   4906 GiB |\n|       from small pool |     13 KiB |    173 MiB |    601 GiB |    601 GiB |\n|---------------------------------------------------------------------------|\n| Active memory         |   8333 KiB |  12642 MiB |   5507 GiB |   5507 GiB |\n|       from large pool |   8320 KiB |  12627 MiB |